# [실거래 · 2026-10-25] 최소 주문금액(5 USDT) 제약 하 실행 방식 비교

라이브 계좌(약 330 USDT)에서 β=30 정책의 주문이 거의 체결되지 않는 원인인 **바이낸스 최소 주문금액(5 USDT)** 을 넣고,
실행 방식을 바꾸면 정책을 얼마나 따라갈 수 있는지 검증·테스트 구간에서 비교한다. **학습 없음** — 배포 중인 재학습 모델
(`models/rt1006/crypto_hrl_pool_v4/agent_beta_30.zip`)만 불러온다.

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 20~40분.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/crypto_min_notional_exec_sim.csv`, 그래프 `crypto_min_notional_exec_sim.png`.

**비교하는 실행 방식** (정책이 원하는 비중 경로는 모두 같다 — 라이브처럼 정책 상태는 체결과 무관하게 스무딩만 따른다)

| 방식 | 내용 |
|---|---|
| ideal | 제약 없음(소수점 금액도 체결) — 상한선 |
| threshold | **현재 라이브**: 목표와의 차이가 5.25 USDT 이상일 때만 주문, 목표가 사실상 0이면 전량 매도 |
| concentrate | 동적 집중: 총 노출은 유지하고 종목당 목표가 15.75 USDT 이상 되도록 상위 n개 종목에만 배분(보유 종목 우대) |
| round_up | 차이가 최소금액의 절반 이상이면 최소금액(5.25 USDT)으로 올려서 주문 |
| stochastic | 확률적 반올림: 차이/최소금액의 확률로 최소금액 주문 |

공통 조건: 거래비용 0.1%, 매도 먼저·매수는 현금 한도, 5 USDT 미만 잔량은 팔 수 없음, 처음엔 전액 현금에서 시작.
운용금 330 / 800 / 2,000 USDT로 각각 실행한다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


## 1. 정책 목표 비중 경로 (라이브 방식)

라이브(`crypto/inference/policy_v4.py`)는 `p ← (1-α)·p + α·목표`를 정책 내부 상태로 들고, 관측에도 이 `p`를 넣는다(가격 변동에 따른 drift 없음).
체결은 정책 관측에 영향을 주지 않으므로 경로를 한 번만 계산해 모든 실행 방식에 공통으로 쓴다.
참고로 원본 백테스트(`run_beta_smoothed` — 관측에 drift된 실제 비중, 등가중 시작)의 성과도 같이 출력해 기준을 확인한다.

In [ ]:
deployed_model = PPO.load(str(DEPLOYED_AGENT_PATH))


def policy_path(model, buckets, alpha=ALPHA, top_k=TOP_K):
    """라이브와 같은 재귀로 정책이 원하는 비중 경로를 계산한다. 반환: W(비중), R(수익률), T(거래 가능), RHO."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    env.reset()
    p = np.zeros(env.n_assets, dtype=np.float32)  # 라이브처럼 전액 현금에서 시작
    W, R, T, RHO = [], [], [], []
    for t, b in enumerate(buckets[:-1]):         # run_beta_smoothed와 같은 스텝 수
        env.t = t
        obs = np.concatenate([env._asset_emb(b), p]).astype(np.float32)
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, _ = env.action_to_weights(a)
        p = ((1 - alpha) * p + alpha * tw).astype(np.float32)
        W.append(p.copy()); R.append(env._asset_ret(b)); T.append(env._tradable(b)); RHO.append(rho)
    return np.array(W), np.array(R, dtype=np.float64), np.array(T), np.array(RHO)


paths, ref_rows = {}, {}
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    print(f'{split}: 정책 경로 계산 중... ({len(bk):,}스텝)')
    paths[split] = policy_path(deployed_model, bk)
    vals, turns = run_beta_smoothed(deployed_model, ALPHA, bk)
    ref_rows[split] = perf_row(vals, turns)
    W, _, _, RHO = paths[split]
    print(f'  평균 ρ {RHO.mean():.3f} | 평균 목표 노출 {W.sum(1).mean():.3f} | 원본 백테스트 총수익 {ref_rows[split]["총 수익률"]:+.2%}')
print(pd.DataFrame(ref_rows).T.to_string(float_format=lambda x: f'{x:.4f}'))


## 2. 실행 시뮬레이터

`crypto/live_trading_loop_binance.py`의 주문 규칙(최소금액 5 USDT × 여유 5%, 목표 0이면 전량 매도, 매도 먼저)을 달러 단위로 옮겼다.
로컬에서 합성 데이터로 검증: 무제약(ideal)은 비중 기반 백테스트와 수익률이 정확히 같고, 운용금이 매우 크면 threshold도 ideal과 같아진다.

In [ ]:
# 최소 주문금액 실행 시뮬레이터 — 노트북 셀에 그대로 들어간다(TC, TOP_K는 노트북 준비 셀에서 정의됨).
MIN_NOTIONAL = 5.0          # 바이낸스 대부분 USDT 마켓의 NOTIONAL 하한
MARGIN = 1.05               # 라이브 루프의 NOTIONAL_MARGIN과 동일
MV = MIN_NOTIONAL * MARGIN  # 실제로 주문을 내는 최소 금액(5.25 USDT)
PER_NAME = 3 * MV           # 동적 집중에서 종목당 최소 목표 금액
HOLD_BONUS = 1.5            # 동적 집중: 이미 보유 중인 종목은 비중을 이 배수로 쳐서 순위를 매긴다(경계 종목 매매 반복 방지)


def concentrate(w, equity, held=None, per_name=PER_NAME, max_names=10):
    """동적 집중: 총 노출(합)은 유지하고, 종목당 목표가 per_name 이상 되도록 비중 상위 n개에만 다시 나눈다.
    held(보유 중 마스크)가 주어지면 보유 종목을 HOLD_BONUS배로 우대해, 순위 경계에서 통째로 사고팔기를
    반복하며 수수료만 내는 것을 막는다."""
    x = float(w.sum())
    if x <= 0:
        return w
    n = int(np.clip(np.floor(x * equity / per_name), 1, max_names))
    score = w * np.where(held, HOLD_BONUS, 1.0) if held is not None else w
    top = np.argsort(score)[-n:]
    out = np.zeros_like(w)
    out[top] = w[top]
    s = out.sum()
    return out * (x / s) if s > 0 else out


def simulate(W, R, T, capital, mode, tc=0.001, seed=0):
    """달러 단위 실행 시뮬레이션.

    W: (S, N) 정책이 원하는 비중 경로(스무딩 후), R: (S, N) 다음 30분 수익률, T: (S, N) 거래 가능 마스크.
    mode: 'ideal'(무제약) / 'threshold'(현재 라이브) / 'concentrate'(동적 집중) /
          'round_up'(차이가 최소금액의 절반 이상이면 최소금액으로 주문) / 'stochastic'(확률적 반올림).
    순서는 백테스트와 같다: 버킷 b에서 비중을 맞추고(비용 차감) → 그 버킷의 수익률을 반영.
    """
    rng = np.random.default_rng(seed)
    S, N = W.shape
    V = np.zeros(N)              # 종목별 보유 평가액(USDT)
    cash = float(capital)
    nav = [float(capital)]
    gaps, expo = [], []
    n_orders, traded = 0, 0.0
    for t in range(S):
        tr = T[t]
        E = cash + V.sum()
        w_policy = W[t] * tr     # 거래 불가 종목 몫은 현금(백테스트 step_with_weights와 동일)
        w = concentrate(w_policy, E, held=V >= MIN_NOTIONAL) if mode == 'concentrate' else w_policy
        D = E * w
        d = (D - V) * tr
        absd = np.abs(d)
        if mode == 'ideal':
            o = d.copy()
        else:
            o = np.zeros(N)
            big = absd >= MV
            o[big] = d[big]
            sell_all = big & (d < 0) & (D < MV)   # 목표가 사실상 0이면 전량 매도(라이브와 동일)
            o[sell_all] = -V[sell_all]
            small = (~big) & (absd > 1e-12)
            if mode == 'round_up':
                m = small & (absd >= 0.5 * MV)
                o[m] = np.sign(d[m]) * MV
            elif mode == 'stochastic':
                m = small & (rng.random(N) < absd / MV)
                o[m] = np.sign(d[m]) * MV
        o = np.maximum(o, -V)                     # 보유량 이상은 못 판다
        if mode != 'ideal':
            o[(o < 0) & (-o < MIN_NOTIONAL)] = 0  # 5 USDT 미만 매도는 거래소가 거부(팔 수 없는 잔량)
        sells = np.minimum(o, 0)
        cash += -sells.sum() * (1 - tc)
        V += sells
        buys = np.maximum(o, 0)
        budget = cash / (1 + tc)
        if buys.sum() > budget:
            buys *= budget / buys.sum()
            if mode != 'ideal':
                buys[buys < MV] = 0               # 줄인 결과가 최소금액 미만이면 주문 못 함
        cash -= buys.sum() * (1 + tc)
        V += buys
        done = (sells != 0) | (buys != 0)
        n_orders += int(done.sum())
        traded += float(np.abs(sells).sum() + buys.sum())
        E2 = cash + V.sum()
        gaps.append(float(np.abs(V / E2 - w_policy).sum()) if E2 > 0 else 0.0)
        expo.append(float(V.sum() / E2) if E2 > 0 else 0.0)
        V = V * (1 + R[t])
        nav.append(cash + V.sum())
    nav = np.asarray(nav)
    return nav, {
        '평균 노출': float(np.mean(expo)),
        '평균 괴리(L1)': float(np.mean(gaps)),
        '주문 수': n_orders,
        '하루 주문 수': n_orders / (S / 48),
        '누적 거래대금/운용금': traded / capital,
    }


## 3. 운용금 × 실행 방식 비교

In [ ]:
%%time
CAPITALS = [330, 800, 2000]
MODES = ['ideal', 'threshold', 'concentrate', 'round_up', 'stochastic']
rows, navs = [], {}
for split in ['valid', 'test']:
    W, R, T, _ = paths[split]
    for cap in CAPITALS:
        for mode in MODES:
            if mode == 'ideal' and cap != CAPITALS[0]:
                continue  # 무제약은 금액과 무관
            nav, info = simulate(W, R, T, cap, mode, tc=TC)
            perf = compute_perf(nav)
            rows.append({'구간': split, '운용금': cap if mode != 'ideal' else '무관', '방식': mode,
                         '총 수익률': perf['총 수익률'], 'Sharpe': perf['Sharpe'], 'MDD': perf['MDD'], **info})
            navs[(split, cap, mode)] = nav
res = pd.DataFrame(rows)
pd.set_option('display.width', 200)
fmt = {'총 수익률': '{:+.2%}', 'Sharpe': '{:.3f}', 'MDD': '{:.2%}', '평균 노출': '{:.1%}', '평균 괴리(L1)': '{:.3f}',
       '하루 주문 수': '{:.1f}', '누적 거래대금/운용금': '{:.1f}'}
show = res.copy()
for k, f in fmt.items():
    show[k] = show[k].map(f.format)
print(show.drop(columns=['주문 수']).to_string(index=False))
res.to_csv(CRYPTO_DIR / 'crypto_min_notional_exec_sim.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: crypto_min_notional_exec_sim.csv')


## 4. 그래프와 요약

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for col, split in enumerate(['valid', 'test']):
    for row, cap in enumerate([330, 2000]):
        ax = axes[row, col]
        ax.plot(navs[(split, 330, 'ideal')] / 330, label='ideal (no constraint)', color='black', lw=1.2)
        for mode, c in [('threshold', 'tab:red'), ('concentrate', 'tab:blue'), ('round_up', 'tab:green')]:
            ax.plot(navs[(split, cap, mode)] / cap, label=mode, color=c, lw=1)
        ax.set_title(f'{split} | capital {cap} USDT'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_min_notional_exec_sim.png', dpi=120)
plt.show()

print('요약 — 운용금 330 USDT에서 무제약 대비:')
for split in ['valid', 'test']:
    base = res[(res['구간'] == split) & (res['방식'] == 'ideal')].iloc[0]
    for mode in ['threshold', 'concentrate', 'round_up', 'stochastic']:
        r = res[(res['구간'] == split) & (res['운용금'] == 330) & (res['방식'] == mode)].iloc[0]
        print(f'  {split:5s} {mode:12s} 수익률 {r["총 수익률"]:+.2%} (무제약 {base["총 수익률"]:+.2%}) | '
              f'노출 {r["평균 노출"]:.1%} (무제약 {base["평균 노출"]:.1%}) | 괴리 {r["평균 괴리(L1)"]:.3f} | 하루 주문 {r["하루 주문 수"]:.1f}')
